In [ ]:
import pandas as pd

# Load isolated Aterio sample CSV
file_path = "../data/interim/aterio_data_sample.csv"
df = pd.read_csv(file_path)

# Keep only columns relevant to your project
keep_cols = [
    "ATERIO_DATA_CENTER_UID",
    "DATA_CENTER_STAGE",
    "STATE_NAME",
    "STATE_CODE",
    "ATERIO_EST_TOT_POWER_CAPACITY_MW",
    "SELECTED_POWER_CAPACITY_MW"
]

dc = df[keep_cols].copy()

# Rename columns to match your other tables
dc = dc.rename(columns={
    "ATERIO_DATA_CENTER_UID": "data_center_id",
    "DATA_CENTER_STAGE": "project_stage",
    "STATE_NAME": "state_or_region",
    "STATE_CODE": "state_code",
    "ATERIO_EST_TOT_POWER_CAPACITY_MW": "data_center_estimated_power_mw",
    "SELECTED_POWER_CAPACITY_MW": "data_center_selected_power_mw"
})

# Add columns to match the structure of your other datasets
dc["sector"] = "all_sectors"
dc["year"] = 2025   # change if you want a different reference year

# Optional checks
print(dc.head())
print(dc.info())
print(dc["project_stage"].value_counts(dropna=False))

# Create state-level summary for joins
dc_state = (
    dc.groupby(["state_or_region", "sector", "year"], as_index=False)
      .agg(
          data_center_facility_count=("data_center_id", "count"),
          data_center_estimated_power_mw=("data_center_estimated_power_mw", "sum"),
          data_center_selected_power_mw=("data_center_selected_power_mw", "sum")
      )
      .sort_values("data_center_estimated_power_mw", ascending=False)
)
# Save and replace old files
dc.to_csv("../data/processed/data_centers_facility_clean.csv", index=False)
dc_state.to_csv("../data/processed/data_centers_state_summary.csv", index=False)

print("Files saved successfully.")
print(dc_state.head())

                         data_center_id project_stage state_or_region  \
0  14c3ec08-1803-4368-ac7a-fbd879e635eb        Active        Michigan   
1  c5fe22bd-0670-427c-ae03-5674b8e007ef        Active       Wisconsin   
2  302036dd-074a-45de-92fb-15bc3540a4c7     Cancelled        Illinois   
3  28ab55b6-0148-4b98-87ad-c0b4d8dd2a05        Active        New York   
4  83b54f12-8ad5-4318-8fbc-0d87e46a07a7  Announcement    Pennsylvania   

  state_code  data_center_estimated_power_mw  data_center_selected_power_mw  \
0         MI                            1.06                            1.0   
1         WI                            0.60                            0.5   
2         IL                           34.63                           24.0   
3         NY                           23.16                           24.0   
4         PA                           47.70                           72.0   

        sector  year  
0  all_sectors  2025  
1  all_sectors  2025  
2  all_sectors  2

In [2]:
print(sorted(dc_state["state_or_region"].unique()))


['Alabama', 'Arizona', 'California', 'Colorado', 'Connecticut', 'Delaware', 'Florida', 'Georgia', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kentucky', 'Louisiana', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Missouri', 'Nebraska', 'Nevada', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'South Carolina', 'Texas', 'Utah', 'Virginia', 'Washington', 'Wisconsin', 'Wyoming']
